# 01 - Data Mapping
## Extract Evidence from Wiki Dump & Map to FEVER Claims

This notebook walks through:
1. Downloading the FEVER dataset (train + dev + wiki dump)
2. Parsing wiki dump JSONL files
3. Mapping evidence sentences to claims
4. Saving interim CSV files with (claim, evidence, label) triples

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.utils import (
    download_fever_dataset, load_jsonl, clean_text,
    RAW_DIR, WIKI_DIR, INTERIM_DIR, LABEL_MAP
)
from src.mapping import build_wiki_lookup, map_claims_to_evidence

## Step 1: Download FEVER Dataset
This downloads:
- `train.jsonl` (~145K claims)
- `shared_task_dev.jsonl` (~20K claims)
- `wiki-pages.zip` (unzipped to `data/wiki_dump/`)

In [ ]:
# Download FEVER dataset (skip if already downloaded)
download_fever_dataset()

## Step 2: Explore Raw FEVER Data

In [ ]:
# Preview training data
train_data = load_jsonl(RAW_DIR / 'train.jsonl')
print(f'Total training claims: {len(train_data):,}')
print(f'\nSample claim:')
train_data[0]

In [ ]:
# Label distribution
from collections import Counter
labels = Counter(r['label'] for r in train_data)
print('Label Distribution:')
for label, count in labels.most_common():
    print(f'  {label:20s}: {count:,}')

## Step 3: Explore Wiki Dump

In [ ]:
# Build wiki lookup (load a few files for preview)
wiki_lookup = build_wiki_lookup(WIKI_DIR, max_files=3)
print(f'\nLoaded {len(wiki_lookup):,} wiki pages')

# Preview one page
sample_page = list(wiki_lookup.keys())[0]
print(f'\nSample page: {sample_page}')
for sent_id, text in list(wiki_lookup[sample_page].items())[:5]:
    print(f'  [{sent_id}] {text}')

## Step 4: Map Evidence to Claims
Map claims to their evidence sentences from the wiki dump.

Use `max_claims=50000` for a 50K subset, or `None` for all 145K.

In [ ]:
# Map training set (50K subset)
train_df = map_claims_to_evidence(split='train', max_claims=50000)
train_df.head(10)

In [ ]:
# Map dev set (all)
dev_df = map_claims_to_evidence(split='dev')
dev_df.head(10)

## Step 5: Preview Mapped Data

In [ ]:
# Preview some mapped examples
for i, row in train_df.head(5).iterrows():
    print(f'--- Example {i+1} ---')
    print(f'Label:    {row["label_text"]}')
    print(f'Claim:    {row["claim"]}')
    print(f'Evidence: {row["evidence"][:200]}')
    print()

In [ ]:
# Check output files
print('Saved files:')
for f in INTERIM_DIR.glob('*.csv'):
    df = pd.read_csv(f)
    print(f'  {f.name}: {len(df):,} rows')